# Download AIA — 4 eventos de CME (comprimento de onda selecionável)

Baixa dados AIA/SDO nos **4 eventos** usados na análise combinada de 171 Å (`Eclipse/02-Notebooks/main-fft-pca-sun-signals.ipynb`, bloco de PCA/FFT com `ruidos = [ruido, ruido_2, ruido_3, ruido_4]`), para **qualquer canal do AIA** — não só 1700 Å.

| Data | Observação |
|---|---|
| 2011-06-05 | Halo, classe GOES B durante todo o período, sem *flares* associados (evento de referência principal) |
| 2017-04-24 | Evento de CME |
| 2017-04-30 | CME com *flare* associado, **não** halo |
| 2022-10-01 | CME com *flare* associado, halo |

**Canais disponíveis no AIA:** 94, 131, 171, 193, 211, 304, 335 Å (EUV) e 1600, 1700, 4500 Å (UV/visível). Não inclui 2310 Å — esse comprimento de onda não existe no AIA (ver notebook de análise anterior para a discussão sobre isso e sobre SUIT/Aditya-L1 como alternativa).

**Sobre as janelas temporais:** o notebook de download de 171 Å atual (`Sun/sdo_aia_download/main-sun-lightcurves-sunpy.ipynb`) já foi editado várias vezes e não contém mais as células de busca originais para todos os 4 eventos. Para garantir a **mesma cobertura temporal exata** em qualquer canal, as janelas abaixo foram reconstruídas diretamente a partir dos timestamps dos arquivos `.fits` de 171 Å já baixados em `Sun/sdo_aia_download/171/<data>[-no-cme]/`.

**Nota:** a janela SEM CME de 2017-04-30 (01:00–01:50 UT) fica dentro da janela COM CME (00:10–06:15 UT) do mesmo dia — isso é herdado da escolha original dos dados em 171 Å, não uma decisão nova feita aqui.

**Saída:** os `.fits` são salvos em `Sun/sdo_aia_download/<comprimento_de_onda>/<data>[-<comprimento_de_onda>]` e `Sun/sdo_aia_download/<comprimento_de_onda>/<data>-no-cme[-<comprimento_de_onda>]`, seguindo a organização por subpasta já adotada no repositório (`171/`, `1700/`, `304/`, `transit-venus/`). O canal 171 Å foi baixado antes dessa convenção existir e usa nomes de pasta sem sufixo numérico (`2011-06-05`, não `2011-06-05-171`); este notebook respeita essa exceção para não duplicar dados já baixados.

In [ ]:
import os

import matplotlib.pyplot as plt
import sunpy.map
from astropy import units as u
from sunpy.net import Fido, attrs as a


## Selecione o comprimento de onda

Rode a célula abaixo e digite um dos canais do AIA quando solicitado (ex.: `171`, `1700`, `304`).

In [ ]:
# All AIA/SDO channels (Angstrom). 2310 A is intentionally not here: AIA has no
# channel at that wavelength (see the analysis notebook for the SUIT/Aditya-L1 discussion).
AIA_WAVELENGTHS = [94, 131, 171, 193, 211, 304, 335, 1600, 1700, 4500]

# 171 A was downloaded before the "<date>-<wavelength>" folder-suffix convention existed,
# so its folders have no numeric suffix. Every other channel added later follows the suffix
# convention. This keeps new downloads consistent with what is already on disk.
LEGACY_NO_SUFFIX_WAVELENGTHS = {171}


def wavelength_suffix(wavelength):
    """Return the folder-name suffix used for a given AIA wavelength (empty for legacy 171 A)."""
    return "" if wavelength in LEGACY_NO_SUFFIX_WAVELENGTHS else f"-{wavelength}"


def select_wavelength(default=1700):
    """Prompt the user for one of the available AIA wavelengths, retrying on bad input.

    Press Enter with no input to use `default` (1700 A, the last channel used in this project).
    """
    while True:
        raw = input(
            f"Comprimento de onda AIA (Angstrom), opcoes {AIA_WAVELENGTHS} "
            f"[Enter = {default}]: "
        ).strip()
        if raw == "":
            return default
        try:
            wavelength = int(raw)
        except ValueError:
            print("Digite um numero inteiro.")
            continue
        if wavelength not in AIA_WAVELENGTHS:
            print(f"Comprimento de onda invalido. Escolha um destes: {AIA_WAVELENGTHS}")
            continue
        return wavelength


WAVELENGTH_VALUE = select_wavelength()
WAVELENGTH = WAVELENGTH_VALUE * u.angstrom
print(f"Comprimento de onda selecionado: {WAVELENGTH_VALUE} A")


## Eventos e janelas temporais (reconstruídas a partir dos `.fits` de 171 Å já baixados)

In [ ]:
SAMPLE = (10 * 60) * u.s  # one frame every 10 minutes, same cadence as the 171 A downloads

# Time windows reconstructed from the timestamps of the existing 171 A fits files in
# Sun/sdo_aia_download/171/<date>[-no-cme]/, so every channel's coverage matches 171 A exactly.
EVENTS = [
    {
        "date": "2011-06-05",
        "note": "Halo CME, GOES class B throughout, no associated flares (main reference event)",
        "cme_window": ("2011-06-05 04:00", "2011-06-05 08:00"),
        "no_cme_window": ("2011-06-05 01:00", "2011-06-05 03:00"),
    },
    {
        "date": "2017-04-24",
        "note": "CME event",
        "cme_window": ("2017-04-24 00:10", "2017-04-24 03:55"),
        "no_cme_window": ("2017-04-24 04:00", "2017-04-24 06:15"),
    },
    {
        "date": "2017-04-30",
        "note": "CME with associated flare, not halo",
        "cme_window": ("2017-04-30 00:10", "2017-04-30 06:15"),
        "no_cme_window": ("2017-04-30 01:00", "2017-04-30 01:50"),
    },
    {
        "date": "2022-10-01",
        "note": "CME with associated flare, halo",
        "cme_window": ("2022-10-01 11:00", "2022-10-01 15:00"),
        "no_cme_window": ("2022-10-01 02:30", "2022-10-01 05:30"),
    },
]


## Função reutilizável de busca e download

In [ ]:
def download_aia_frames(time_start, time_end, wavelength, sample, output_dir, max_retries=5):
    """
    Search AIA frames via SunPy Fido and download them to output_dir.

    The AIA export server (sdo7.nascom.nasa.gov) is often slow to serve
    individual files and times out under concurrent load, so failed files
    are retried (SunPy's own recommended pattern: pass the Results object
    back into Fido.fetch) with low concurrency instead of failing outright.
    """
    search_result = Fido.search(
        a.Time(time_start, time_end),
        a.Instrument("AIA"),
        a.Wavelength(wavelength),
        a.Sample(sample),
    )
    print(search_result)

    os.makedirs(output_dir, exist_ok=True)
    downloaded_files = Fido.fetch(search_result[0], path=output_dir, max_conn=2)

    retries = 0
    while downloaded_files.errors and retries < max_retries:
        retries += 1
        print(f"{len(downloaded_files.errors)} file(s) failed, retrying ({retries}/{max_retries})...")
        downloaded_files = Fido.fetch(downloaded_files, max_conn=2)

    if downloaded_files.errors:
        print(f"Still {len(downloaded_files.errors)} file(s) failing after {max_retries} retries:")
        print(downloaded_files.errors)
    else:
        print("Download errors: none")

    return downloaded_files


## Download — todos os 4 eventos (COM CME e SEM CME)

In [ ]:
suffix = wavelength_suffix(WAVELENGTH_VALUE)

for event in EVENTS:
    date = event["date"]
    cme_dir = os.path.join("..", "sdo_aia_download", str(WAVELENGTH_VALUE), f"{date}{suffix}")
    no_cme_dir = os.path.join("..", "sdo_aia_download", str(WAVELENGTH_VALUE), f"{date}-no-cme{suffix}")

    print(f"\n=== {date}: {event['note']} ===")

    print("-- with CME --")
    event["files_cme"] = download_aia_frames(
        *event["cme_window"], WAVELENGTH, SAMPLE, cme_dir
    )
    print(f"{len(event['files_cme'])} frames downloaded to {cme_dir}")

    print("-- without CME (reference) --")
    event["files_no_cme"] = download_aia_frames(
        *event["no_cme_window"], WAVELENGTH, SAMPLE, no_cme_dir
    )
    print(f"{len(event['files_no_cme'])} frames downloaded to {no_cme_dir}")


## Conferência rápida (quicklook)

Plota o primeiro frame COM CME de cada evento para checagem visual, salvando os resultados em `Sun/UV/Products/<comprimento_de_onda>A/` (dpi=300), seguindo o padrão de outputs do repositório ECLIPSE.

In [ ]:
def plot_quicklook(fits_path, title, fig_name, output_dir):
    """Quick-look plot of a single AIA frame, saved at 300 dpi."""
    aia_map = sunpy.map.Map(fits_path)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(
        aia_map.data, origin="lower", cmap=aia_map.cmap,
        vmin=0, vmax=aia_map.data.max() * 0.3,
    )
    ax.set_title(title)
    ax.set_xlabel("Pixel X")
    ax.set_ylabel("Pixel Y")
    plt.colorbar(ax.images[0], ax=ax, label="Intensity")
    plt.tight_layout()

    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, f"{fig_name}.png")
    plt.savefig(file_path, dpi=300, bbox_inches="tight")
    print(f"Quicklook saved to: {file_path}")
    plt.show()

    return aia_map


quicklook_dir = os.path.join("Products", f"{WAVELENGTH_VALUE}A")

for event in EVENTS:
    date = event["date"]
    plot_quicklook(
        event["files_cme"][0],
        f"AIA {WAVELENGTH_VALUE} A - with CME - {date}",
        f"quicklook_{WAVELENGTH_VALUE}A_{date}_with-cme",
        quicklook_dir,
    )


## Próximos passos

Os `.fits` baixados aqui (`Sun/sdo_aia_download/<comprimento_de_onda>/<data>[-<comprimento_de_onda>]` e `.../<data>-no-cme[-<comprimento_de_onda>]`, para os 4 eventos) são consumidos pelo notebook de análise **`main-fft-analysis.ipynb`** (também genérico por comprimento de onda), que simula o trânsito de HD 189733 Ab sobre esses frames usando a classe `Estrela(useFits=True, fits_path=...)` do Core do ECLIPSE, e reproduz a análise combinada (curvas de luz, resíduo, PCA e FFT entre os 4 eventos) — exatamente como feito para 171 Å em `main-fft-pca-sun-signals.ipynb`.